# GasGuard: Dataset Explorer & Generator
### Step 1 of the Machine Learning Pipeline

**Student:** WALP Harsha  
**Registration No:** D/ENG/24/0095/ET  

This notebook generates the synthetic historical cylinder usage dataset, explores the features, prints statistical descriptions, and visualizes the depletion cycles and cooking habit trends.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("muted")
print("✓ Libraries loaded.")

## 2. Generate 180-Day Gas Cylinder Consumption Dataset
A **5kg gas tank** takes **2 months (60 days)** to empty. This represents an average daily consumption of **0.0833 kg/day**. We simulate weekend fluctuations and daily noise over a 180-day timeline (3 full cycles).

In [ ]:
print("📡 Generating dataset...")
np.random.seed(42)

dates = pd.date_range(start='2026-01-01', end='2026-06-29', freq='D')
n_days = len(dates)

base_consumption = 0.0833
weekly_variation = [0.07, 0.075, 0.08, 0.09, 0.10, 0.095, 0.073]
noise = np.random.normal(0, 0.005, n_days)

gas_weight = []
daily_usage = []

for i in range(n_days):
    day_of_week = i % 7
    usage = base_consumption + (weekly_variation[day_of_week] - 0.0833) + noise[i]
    usage = max(0.04, min(0.15, usage))
    daily_usage.append(usage)
    
    if i == 0:
        current_weight = 5.0
    else:
        current_weight = current_weight - usage
    
    if current_weight < 0.1 or (i > 0 and i % 60 == 0):
        current_weight = 5.0
    
    gas_weight.append(max(0.0, current_weight))

df = pd.DataFrame({
    'date': dates,
    'gas_weight_kg': gas_weight,
    'daily_usage_kg': daily_usage,
    'cylinder_type': '5kg Standard'
})

# Feature engineering
df['gas_percentage'] = (df['gas_weight_kg'] / 5.0) * 100
df['days_since_refill'] = range(n_days)
df['day_of_week'] = df['date'].dt.dayofweek
df['weekend'] = (df['day_of_week'] >= 5).astype(int)
df['consumption_rate'] = df['daily_usage_kg'] / df['gas_percentage']
df['usage_velocity'] = df['daily_usage_kg'].diff().fillna(0)
df['rolling_avg_7'] = df['daily_usage_kg'].rolling(7).mean().fillna(df['daily_usage_kg'].mean())

# Target Variable calculation
df['days_remaining'] = 0
for i in range(len(df)):
    remaining = df.loc[i, 'gas_weight_kg']
    if remaining > 0.1:
        future_days = 0
        temp_weight = remaining
        for j in range(i+1, min(i+90, len(df))):
            temp_weight -= df.loc[j, 'daily_usage_kg']
            if temp_weight <= 0.1:
                future_days = j - i
                break
        df.loc[i, 'days_remaining'] = future_days if future_days > 0 else 1
    else:
        df.loc[i, 'days_remaining'] = 0

df['refill_price_lkr'] = 1910
df['price_per_kg_lkr'] = 382
df['estimated_monthly_cost'] = df['daily_usage_kg'] * 30 * df['price_per_kg_lkr']

df.to_csv('train_dataset.csv', index=False)
print("✓ Dataset generated and saved as train_dataset.csv.")

## 3. Explore & Inspect the Dataset Structure

In [ ]:
print("📋 Dataset Shape:", df.shape)
print("\n📋 First 10 rows of the Dataset:")
df.head(10)

In [ ]:
print("📋 Statistical Summary:")
df.describe()

## 4. Dataset Visualizations

In [ ]:
plt.figure(figsize=(12, 5), dpi=150)
plt.plot(df['date'], df['gas_weight_kg'], color='#00796B', linewidth=2.5)
plt.fill_between(df['date'], df['gas_weight_kg'], color='#00897B', alpha=0.15)
plt.axhline(0.1, color='r', linestyle='--', label='Empty limit (0.1 kg)')
plt.title('Gas Cylinder Weight Depletion Timeline (3 Full Refill Cycles)')
plt.xlabel('Date')
plt.ylabel('Gas Weight (kg)')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5), dpi=150)
day_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
weekly_avg = df.groupby('day_of_week')['daily_usage_kg'].mean().reset_index()
weekly_avg['day_name'] = weekly_avg['day_of_week'].map(day_names)

colors = ['#64748B'] * 5 + ['#F43F5E', '#F43F5E']
plt.bar(weekly_avg['day_name'], weekly_avg['daily_usage_kg'], color=colors)
plt.title('Average Cooking Usage by Day of Week (Weekend Peak highlighted)')
plt.xlabel('Day')
plt.ylabel('Usage (kg)')
plt.show()